# 03 · Construção do dataset

Aplica o processamento do notebook 02 aos 6.695 KOIs, faz a partição em treino, validação e teste por estrela e aplica o SMOTE ao conjunto de treino. O resultado é o arquivo `dataset.npz`, usado por todos os notebooks seguintes.

O download de milhares de curvas leva horas, e a sessão gratuita do Colab é interrompida periodicamente. Por isso o notebook tem duas partes:
- **Parte A:** baixa as curvas e grava as visões de cada KOI em um arquivo separado no Drive. Ao ser executada de novo, pula os KOIs já processados;
- **Parte B:** executada depois que a Parte A termina, junta as visões, faz a partição e aplica o SMOTE.

Não precisa de GPU.

**Versões dos dados.** As visões da rodada final ficam em `views_v2/`. A pasta `views/` guarda as da primeira rodada, geradas com o corte de *outliers* simétrico (ver notebook 10 e Apêndice B do TCC).

## 1. Configuração

In [ ]:
!pip install -q lightkurve==2.6.0

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.stats import binned_statistic
from google.colab import drive

drive.mount('/content/drive')

# --- Parametros (identicos ao piloto) ---
SIGMA_OUTLIER = 6.0   # so pontos ACIMA de 6 sigma (sigma_lower=inf); ver Notebook 10
GLOBAL_BINS   = 2001
LOCAL_BINS    = 201
K_DURATIONS   = 4
SEED          = 42

DATA_DIR  = '/content/drive/MyDrive/TCC-Exoplanetas'
# v2 (nov/2026): corte de outliers so no lado superior. A pasta 'views' (v1, corte simetrico,
# que apagava o fundo de transitos profundos) fica preservada para comparacao.
VIEWS_DIR = f'{DATA_DIR}/views_v2'   # um .npz por KOI
os.makedirs(VIEWS_DIR, exist_ok=True)

labels = pd.read_csv(f'{DATA_DIR}/koi_labels.csv')
print('KOIs a processar:', len(labels))

/usr/local/lib/python3.13/dist-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


Mounted at /content/drive
KOIs a processar: 6695


In [ ]:
# Teste de fumaca: confirma que ESCREVE e LE de volta no Drive.
# Rode uma vez antes de disparar a Parte A. Se falhar aqui, nada seria salvo.
_test = f'{VIEWS_DIR}/_teste_escrita.npz'
np.savez_compressed(_test, x=np.arange(5))
assert os.path.exists(_test), 'FALHOU: nao gravou no Drive! Verifique a montagem e o caminho.'
print('OK — escreveu e leu de volta:', np.load(_test)['x'])
print('Destino confirmado:', VIEWS_DIR)
os.remove(_test)
print('Pode rodar a Parte A com seguranca.')

OK — escreveu e leu de volta: [0 1 2 3 4]
Destino confirmado: /content/drive/MyDrive/TCC-Exoplanetas/views_v2
Pode rodar a Parte A com seguranca.


## 2. Funções

As mesmas do notebook 02. Cada estrela é baixada uma única vez, e as visões de todos os seus KOIs são geradas a partir da mesma curva.

In [ ]:
import shutil, tempfile

def median_bin(x, y, num_bins, x_min, x_max):
    stat, edges, _ = binned_statistic(x, y, statistic='median', bins=num_bins, range=(x_min, x_max))
    centers = 0.5 * (edges[:-1] + edges[1:])
    nans = np.isnan(stat)
    if nans.all():
        return stat
    if nans.any():
        stat[nans] = np.interp(centers[nans], centers[~nans], stat[~nans])
    return stat


def normalize_view(v):
    v = v - np.median(v)
    depth = np.abs(np.min(v))
    return v / depth if depth > 0 else v


def baixar_curva(kepid):
    # Download PDCSAP (cadencia longa, todos os quarteis) + limpeza. Uma vez por ESTRELA.
    search = lk.search_lightcurve(f'KIC {kepid}', mission='Kepler', author='Kepler', cadence='long')
    if len(search) == 0:
        return None
    tmp = tempfile.mkdtemp()            # pasta propria por estrela: seguro em paralelo
    try:
        lcc = search.download_all(flux_column='pdcsap_flux', download_dir=tmp)
        return lcc.stitch().remove_nans().remove_outliers(sigma=SIGMA_OUTLIER, sigma_lower=np.inf)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)   # nao acumula FITS no disco do Colab


def gerar_visoes(lc, row):
    # Visoes Global e Local de UM KOI a partir da curva da estrela (identico ao Notebook 02).
    period   = float(row.koi_period)
    t0       = float(row.koi_time0bk)
    dur_days = float(row.koi_duration) / 24.0

    mask = lc.create_transit_mask(period=period, transit_time=t0, duration=dur_days)
    flat = lc.flatten(window_length=901, mask=mask)

    folded = flat.fold(period=period, epoch_time=t0)
    phase = np.asarray(folded.phase.value)
    flux  = np.asarray(folded.flux.value)
    ok = np.isfinite(phase) & np.isfinite(flux)
    phase, flux = phase[ok], flux[ok]

    g = normalize_view(median_bin(phase, flux, GLOBAL_BINS, -period / 2, period / 2))
    half_local = min(K_DURATIONS * dur_days, period / 2)
    l = normalize_view(median_bin(phase, flux, LOCAL_BINS, -half_local, half_local))
    return g, l

## Parte A · Extração em lote

Pode ser executada várias vezes, em sessões diferentes, e sempre continua de onde parou. Cada arquivo é gravado primeiro com um nome temporário e só depois renomeado, para que uma interrupção não deixe arquivos incompletos.

Como o tempo é dominado pela espera do servidor do MAST, várias estrelas são baixadas em paralelo (`N_WORKERS`). Se houver muitas falhas de conexão, reduza para 4.

In [ ]:
import sys, time, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import astroquery.mast                       # importa antes de mexer nos loggers (ver abaixo)
from astroquery import log as log_astroquery
from astropy import log as log_astropy

N_WORKERS = 8

# Silencia avisos de unidades FITS e as mensagens 'Downloading URL ...' (milhares de linhas
# travariam o navegador). O progresso continua sendo impresso a cada 50 estrelas.
# Usa os loggers dos proprios pacotes: criar um logger generico com o nome 'astroquery' antes
# de o pacote ser importado quebra o astroquery ("'Logger' object has no attribute '_set_defaults'").
warnings.filterwarnings('ignore')
log_astroquery.setLevel('ERROR')
log_astropy.setLevel('ERROR')

def ja_feito(kepoi):
    return os.path.exists(f'{VIEWS_DIR}/{kepoi}.npz')

todo = labels[~labels['kepoi_name'].map(ja_feito)].reset_index(drop=True)
estrelas = list(todo.groupby('kepid'))
print(f'Ja processados: {len(labels) - len(todo)} / {len(labels)} KOIs')
print(f'Restantes: {len(todo)} KOIs em {len(estrelas)} estrelas  |  {N_WORKERS} downloads em paralelo')

def processar_estrela(kepid, grupo):
    falhas_e = []
    try:
        lc = baixar_curva(int(kepid))
    except Exception as e:
        return [(k, 'download: ' + str(e)[:110]) for k in grupo['kepoi_name']]
    if lc is None:
        return [(k, 'sem curvas') for k in grupo['kepoi_name']]
    for _, row in grupo.iterrows():
        try:
            g, l = gerar_visoes(lc, row)
            # grava em arquivo temporario e renomeia: uma interrupcao no meio da escrita
            # nunca deixa um .npz corrompido (que seria contado como 'ja feito')
            destino = f"{VIEWS_DIR}/{row['kepoi_name']}.npz"
            with open(destino + '.part', 'wb') as fh:
                np.savez_compressed(
                    fh,
                    global_view=g.astype('float32'),
                    local_view=l.astype('float32'),
                    label=np.int8(row['label']),
                    kepid=np.int64(row['kepid']),
                )
            os.replace(destino + '.part', destino)
        except Exception as e:
            falhas_e.append((row['kepoi_name'], str(e)[:120]))
    return falhas_e

# Teste rapido: a primeira estrela roda fora do paralelo. Se ela falhar, a celula PARA e mostra o
# erro, em vez de registrar milhares de falhas em silencio.
assert ('baixar_curva' in globals() and 'gerar_visoes' in globals()), (
    'Rode antes a celula de funcoes (secao 2) da versao atual do notebook.')
k0, g0 = estrelas[0]
f0 = processar_estrela(k0, g0)
if len(f0) == len(g0):
    raise RuntimeError(f'Falha ja na primeira estrela (KIC {k0}): {f0[0][1]}')
print(f'Teste OK (KIC {k0}). Iniciando o paralelo...')
estrelas = estrelas[1:]

falhas, t_ini = [], time.time()
saida_real = sys.stdout                  # o progresso vai para a celula...
sys.stdout = open(os.devnull, 'w')       # ...e os prints internos do download, para lugar nenhum
try:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futuros = [ex.submit(processar_estrela, k, g) for k, g in estrelas]
        for n, fut in enumerate(as_completed(futuros), 1):
            falhas.extend(fut.result())
            if n >= 100 and len(falhas) >= 0.9 * n:
                ex.shutdown(cancel_futures=True)
                raise RuntimeError(f'Mais de 90% de falhas nas primeiras {n} estrelas. '
                                   f'Ultimo motivo: {falhas[-1][1]}')
            if n % 50 == 0 or n == len(futuros):
                dt = time.time() - t_ini
                restante = (len(futuros) - n) * dt / n
                print(f'  {n}/{len(futuros)} estrelas | {60 * n / dt:.0f} estrelas/min | '
                      f'falta ~{restante / 3600:.1f} h | falhas: {len(falhas)}',
                      file=saida_real, flush=True)
finally:
    sys.stdout.close()
    sys.stdout = saida_real

print(f'\nRodada concluida. Falhas: {len(falhas)}')
if falhas:
    pd.DataFrame(falhas, columns=['kepoi_name', 'motivo']).to_csv(
        f'{DATA_DIR}/falhas_v2.csv', index=False)
    print('Registradas em falhas_v2.csv (serao retentadas na proxima rodada)')

Ja processados: 6692 / 6695 KOIs
Restantes: 3 KOIs em 2 estrelas  |  8 downloads em paralelo
Teste OK (KIC 4285087). Iniciando o paralelo...
  1/1 estrelas | 7 estrelas/min | falta ~0.0 h | falhas: 0

Rodada concluida. Falhas: 0


In [ ]:
# Diagnostico de progresso (rode a qualquer momento)
n_feitos = len(glob.glob(f'{VIEWS_DIR}/*.npz'))
print(f'Progresso total: {n_feitos} / {len(labels)}  ({n_feitos / len(labels):.1%})')

Progresso total: 6695 / 6695  (100.0%)


## Parte B · Montagem, partição e SMOTE

Executar só depois que a Parte A tiver processado todos os KOIs.

### Arquivamento da primeira rodada

Esta célula foi usada na transição entre as duas rodadas do experimento. Os notebooks de treino pulam os modelos que já existem em `modelos/`; por isso, antes de gerar o novo `dataset.npz`, os arquivos da primeira rodada (partição por KOI) foram movidos para `v1_particao_por_koi/`. Assim o retreino era forçado e os resultados antigos ficavam preservados para comparação. Nessa transição, o notebook 10 foi executado antes desta célula, porque usa as predições da primeira rodada.

Em uma execução do zero, esta célula deve ser pulada: ela exige os arquivos da primeira rodada e interrompe a execução se não os encontrar.

In [ ]:
import shutil

V1_DIR = f'{DATA_DIR}/v1_particao_por_koi'
ARTEFATOS = ['dataset.npz', 'modelos', 'figuras', 'resultados_teste.csv', 'predicoes_teste.csv',
             'atencao_global_quantificacao.csv', 'interpretabilidade_concentracao.csv']

if not os.path.exists(f'{DATA_DIR}/diagnostico_outliers.csv') and not os.path.exists(V1_DIR):
    raise RuntimeError('Rode o Notebook 10 antes de arquivar (ele usa o predicoes_teste.csv antigo).')

os.makedirs(V1_DIR, exist_ok=True)
for a in ARTEFATOS:
    origem, destino = f'{DATA_DIR}/{a}', f'{V1_DIR}/{a}'
    if os.path.exists(origem) and not os.path.exists(destino):
        shutil.move(origem, destino)
        print('arquivado:', a)
    elif os.path.exists(destino):
        print('ja arquivado:', a)
print('Conteudo de v1_particao_por_koi/:', sorted(os.listdir(V1_DIR)))

arquivado: dataset.npz
arquivado: modelos
arquivado: figuras
arquivado: resultados_teste.csv
arquivado: predicoes_teste.csv
arquivado: atencao_global_quantificacao.csv
arquivado: interpretabilidade_concentracao.csv
Conteudo de v1_particao_por_koi/: ['atencao_global_quantificacao.csv', 'dataset.npz', 'figuras', 'interpretabilidade_concentracao.csv', 'modelos', 'predicoes_teste.csv', 'resultados_teste.csv']


In [ ]:
files = sorted(glob.glob(f'{VIEWS_DIR}/*.npz'))
print('Arquivos .npz encontrados:', len(files))

G, L, Y, KID, NOME = [], [], [], [], []
ignorados = 0
for f in files:
    d = np.load(f)
    # pula arquivos que nao sejam Visoes (ex.: _teste_escrita / marcador)
    if 'global_view' not in d.files:
        ignorados += 1
        continue
    g, l = d['global_view'], d['local_view']
    # descarta visoes com NaN/inf (alvo problematico)
    if not (np.isfinite(g).all() and np.isfinite(l).all()):
        ignorados += 1
        continue
    G.append(g); L.append(l)
    Y.append(int(d['label'])); KID.append(int(d['kepid']))
    NOME.append(os.path.basename(f)[:-4])

Xg = np.stack(G); Xl = np.stack(L)
y  = np.array(Y); kid = np.array(KID); nome = np.array(NOME)
print(f'Visoes validas: {len(y)}  |  ignorados: {ignorados}')
print('Xg:', Xg.shape, '| Xl:', Xl.shape, '| positivos:', y.mean().round(3))
print(f'Estrelas distintas: {len(np.unique(kid))}  |  estrelas com mais de um KOI: '
      f'{(np.unique(kid, return_counts=True)[1] > 1).sum()}')

Arquivos .npz encontrados: 6695
Visoes validas: 6695  |  ignorados: 0
Xg: (6695, 2001) | Xl: (6695, 201) | positivos: 0.408
Estrelas distintas: 5828  |  estrelas com mais de um KOI: 583


### Partição por estrela

Uma mesma estrela pode ter vários KOIs, que compartilham a mesma curva de luz. Se KOIs da mesma estrela caem em conjuntos diferentes, o modelo pode encontrar no teste uma curva que já viu no treino, e as métricas ficam otimistas. Na primeira rodada, a partição era feita por KOI e tinha esse problema (ver notebook 10).

A partição usa `StratifiedGroupKFold` com 10 dobras: todos os KOIs de uma estrela ficam na mesma dobra, e cada dobra mantém aproximadamente a proporção de classes. Uma dobra forma o teste, outra a validação e as oito restantes o treino. Os nomes dos KOIs de cada conjunto ficam gravados no `dataset.npz`.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

idx = np.arange(len(y))
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
dobras = [d_te for _, d_te in sgkf.split(idx, y, groups=kid)]
te, va = dobras[0], dobras[1]
tr = np.setdiff1d(idx, np.concatenate([te, va]))

# Garantia: nenhuma estrela aparece em mais de um conjunto
e_tr, e_va, e_te = set(kid[tr]), set(kid[va]), set(kid[te])
assert not (e_tr & e_va) and not (e_tr & e_te) and not (e_va & e_te), 'Estrela compartilhada entre conjuntos!'
assert len(tr) + len(va) + len(te) == len(y)

for nome_c, s in [('treino', tr), ('validacao', va), ('teste', te)]:
    print(f'{nome_c:10s} {len(s):5d} KOIs ({len(s) / len(y):.1%}) | positivos {y[s].mean():.3f} | '
          f'estrelas {len(np.unique(kid[s]))}')
print('OK — nenhuma estrela compartilhada entre treino, validacao e teste.')

treino      5342 KOIs (79.8%) | positivos 0.409 | estrelas 4661
validacao    683 KOIs (10.2%) | positivos 0.425 | estrelas 583
teste        670 KOIs (10.0%) | positivos 0.379 | estrelas 584
OK — nenhuma estrela compartilhada entre treino, validacao e teste.


In [ ]:
# SMOTE apenas no treino (evita vazamento para val/teste)
!pip install -q imbalanced-learn==0.14.2
from imblearn.over_sampling import SMOTE

Xtr = np.hstack([Xg[tr], Xl[tr]])          # concatena as duas visoes
sm = SMOTE(random_state=SEED)
Xtr_res, ytr_res = sm.fit_resample(Xtr, y[tr])
Xg_tr = Xtr_res[:, :GLOBAL_BINS]
Xl_tr = Xtr_res[:, GLOBAL_BINS:]
print(f'treino antes: {len(tr)} (pos {y[tr].mean():.2f})  ->  depois SMOTE: {len(ytr_res)} (pos {ytr_res.mean():.2f})')

treino antes: 5342 (pos 0.41)  ->  depois SMOTE: 6312 (pos 0.50)


In [ ]:
np.savez_compressed(
    f'{DATA_DIR}/dataset.npz',
    Xg_train=Xg_tr.astype('float32'), Xl_train=Xl_tr.astype('float32'), y_train=ytr_res.astype('int8'),
    Xg_val=Xg[va].astype('float32'),  Xl_val=Xl[va].astype('float32'),  y_val=y[va].astype('int8'),
    Xg_test=Xg[te].astype('float32'), Xl_test=Xl[te].astype('float32'), y_test=y[te].astype('int8'),
    # identidade dos KOIs (o treino guarda so os originais; as linhas do SMOTE sao sinteticas)
    koi_train=nome[tr], koi_val=nome[va], koi_test=nome[te],
    kepid_train=kid[tr], kepid_val=kid[va], kepid_test=kid[te],
)
print('Salvo:', f'{DATA_DIR}/dataset.npz')
print('Pronto para o Notebook 05 (treino).')

Salvo: /content/drive/MyDrive/TCC-Exoplanetas/dataset.npz
Pronto para o Notebook 05 (treino).


## Próximo passo

O notebook 04 define as três arquiteturas.